# Assignment 2 — Hyperparameter Tuning

**Purpose:** Improve one LSTM through causal features, optimization recipes, and paired rolling selection.

This notebook is an executable evidence view over the shared local pipeline. The
command-line runner and notebook call the same stage implementation, preventing code
drift. Completed artifacts are reused; no Google Colab or Google Drive runtime is used.


In [1]:
# Local WSL project bootstrap. Dependency installation belongs to the
# locked project environment and is intentionally not repeated in notebooks.
from pathlib import Path
import json
import os
import platform
import sys

candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [candidate, *candidate.parents] if (path / "shared").exists()),
    candidate,
)
if not (PROJECT_ROOT / "shared").exists():
    raise FileNotFoundError(
        "Open this notebook from the local odl-assignment project root."
    )
os.environ["ODL_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.config import PATHS
from shared.runtime import RuntimeOptions
from shared.workflow import run_stage

OPTIONS = RuntimeOptions.from_environment()
print("Project root:", PROJECT_ROOT)
print("Run mode:", OPTIONS.mode)
print("Device policy:", OPTIONS.device)
print("WSL distribution:", os.getenv("WSL_DISTRO_NAME", "not detected"))


I0000 00:00:1786592294.958407  642676 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1786592299.370636  642676 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Project root: /home/nicholas/code/odl-assignment
Run mode: smoke
Device policy: gpu
WSL distribution: Ubuntu-24.04


In [2]:
# Run the shared stage or load its verified completed result.
STAGE_SUMMARY = run_stage("tune", options=OPTIONS)
STAGE_SUMMARY


{'architecture': 'lstm',
 'selected_feature_policy': 'multivariate_dynamics',
 'selected_recipe': 'recipe_02',
 'recipe_validation_fits': 6,
 'candidate_count': 2,
 'validation_fits': 2,
 'selected_candidate': 'candidate_01',
 'formulation': 'residual',
 'validation_mae': 11.158179018198776,
 'validation_gate_passed': True,
 'validation_gate_bypassed_for_smoke': True,
 'epoch_budget': 1,
 'training_seconds': 20.95886004299973,
 'run_mode': 'smoke'}

In [3]:
# Record the exact runtime used for this output when preflight evidence exists.
environment_path = PATHS.environment_artifacts / f"environment_{OPTIONS.mode}.json"
ENVIRONMENT = (
    json.loads(environment_path.read_text(encoding="utf-8"))
    if environment_path.exists()
    else {"status": "preflight evidence not yet generated"}
)
ENVIRONMENT


{'captured_at_utc': '2026-08-13T03:30:49.802312+00:00',
 'project_root': '/home/nicholas/code/odl-assignment',
 'platform': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39',
 'system': 'Linux',
 'release': '6.6.87.2-microsoft-standard-WSL2',
 'python': '3.12.3',
 'wsl_distribution': 'Ubuntu-24.04',
 'device': 'gpu',
 'tensorflow_gpu_devices': ['/physical_device:GPU:0'],
 'batch_size': 256,
 'mixed_precision': False,
 'nvidia_smi': {'available': True,
  'name': 'NVIDIA GeForce RTX 3050 Laptop GPU',
  'driver_version': '591.74',
  'memory.total': '4096',
  'memory.used': '731',
  'utilization.gpu': '7',
  'temperature.gpu': '64',
  'power.draw': '12.46'},
 'packages': {'tensorflow': '2.21.0',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'pyarrow': '25.0.0',
  'scikit-learn': '1.9.0',
  'matplotlib': '3.11.1',
  'seaborn': '0.13.2'},
 'git': {'available': True,
  'revision': '1ac2e21f1505f3b7ad173781d2e1a0293b098576',
  'dirty': True},
 'experiment_config': {'methodology_versi

## Staged, focused tuning evidence

**Decision.** First compare six preregistered loss/optimizer recipes using candidate
13's promising residual architecture, then evaluate 12 deterministic local candidates
across three rolling folds and seeds 42/52. The local search contains the basic and
candidate-13 anchors plus ten balanced configurations. Random search is efficient when
only some dimensions strongly affect performance
([Bergstra and Bengio](https://www.jmlr.org/papers/v13/bergstra12a.html)). **Alternatives.**
A broad second global search would spend scarce GPU time in regions already shown to be
weak. **Trade-off.** The focused design costs 108 recipe/search fits after the 18-fit
feature experiment, but every decision remains based on rolling MAE. The frozen
benchmark is never read during selection.


In [4]:
import pandas as pd

suffix = OPTIONS.artifact_suffix
recipe_trials = pd.read_csv(
    PATHS.assignment2_tables / f"optimization_recipe_results{suffix}.csv"
)
recipe_summary = pd.read_csv(
    PATHS.assignment2_tables / f"optimization_recipe_summary{suffix}.csv"
)
recipe_audit = pd.read_csv(
    PATHS.assignment2_tables / f"optimization_recipe_paired_audit{suffix}.csv"
)
trials = pd.read_csv(
    PATHS.assignment2_tables / f"focused_lstm_search_results{suffix}.csv"
)
summary = pd.read_csv(
    PATHS.assignment2_tables / f"focused_lstm_search_summary{suffix}.csv"
)
paired_audit = pd.read_csv(
    PATHS.assignment2_tables / f"focused_lstm_paired_audit{suffix}.csv"
)
best_hyperparameters = json.loads(
    (PATHS.models / f"best_hyperparameters{suffix}.json").read_text(encoding="utf-8")
)
display(recipe_summary)
display(recipe_audit)
display(recipe_trials)
display(summary)
display(paired_audit)
display(trials)
display(best_hyperparameters)


,candidate_id,mean_mae,std_mae,folds,mean_seed_std_mae,min_seeds_per_fold,max_seeds_per_fold,evaluations,mean_best_epoch,median_best_epoch,parameters,num_layers,formulation,se_mae,symmetric_loss_rank,optimizer_rank,mechanism_rank
0,recipe_02,11.158179,NaN,1,0.0,1,1,1,1.0,1.0,239745,2,residual,0.0,0,0,2
1,recipe_05,11.158179,NaN,1,0.0,1,1,1,1.0,1.0,239745,2,residual,0.0,0,1,3
2,recipe_01,11.158333,NaN,1,0.0,1,1,1,1.0,1.0,239745,2,residual,0.0,0,0,2
3,recipe_00,11.163934,NaN,1,0.0,1,1,1,1.0,1.0,239745,2,residual,0.0,0,0,0
4,recipe_03,11.191638,NaN,1,0.0,1,1,1,1.0,1.0,239745,2,residual,0.0,0,0,2
5,recipe_04,11.250752,NaN,1,0.0,1,1,1,1.0,1.0,239745,2,residual,0.0,1,0,2


,candidate_id,raw_winner,mean_paired_degradation,paired_standard_error,paired_one_se_eligible,paired_folds
0,recipe_02,recipe_02,0.000000,0.0,True,1
1,recipe_05,recipe_02,0.000000,0.0,True,1
2,recipe_01,recipe_02,0.000154,0.0,False,1
3,recipe_00,recipe_02,0.005755,0.0,False,1
4,recipe_03,recipe_02,0.033459,0.0,False,1
5,recipe_04,recipe_02,0.092573,0.0,False,1


,candidate_id,formulation,units,dense_units,dropout,learning_rate,l2_strength,num_layers,huber_delta,loss_name,...,source_fingerprint,n,mae,rmse,r2,smape_percent,best_epoch,epochs_recorded,parameters,training_seconds
0,recipe_00,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,8469,11.163934,22.323607,0.897737,21.153089,1,1,239745,14.647183
1,recipe_01,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,8469,11.158333,22.332920,0.897652,21.135519,1,1,239745,19.146354
2,recipe_02,residual,128,64,0.2,0.00056,0.0001,2,2.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,8469,11.158179,22.323596,0.897738,21.132148,1,1,239745,18.453911
3,recipe_03,residual,128,64,0.2,0.00056,0.0001,2,2.0,mae,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,8469,11.191638,22.384579,0.897178,21.181885,1,1,239745,19.706718
4,recipe_04,residual,128,64,0.2,0.00056,0.0001,2,2.0,pinball,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,8469,11.250752,22.443425,0.896637,21.222289,1,1,239745,17.254270
5,recipe_05,residual,128,64,0.2,0.00056,0.0001,2,2.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,8469,11.158179,22.323596,0.897738,21.132148,1,1,239745,18.415825


,candidate_id,mean_mae,std_mae,folds,mean_seed_std_mae,min_seeds_per_fold,max_seeds_per_fold,evaluations,mean_best_epoch,median_best_epoch,parameters,num_layers,formulation,se_mae
0,candidate_01,11.158179,NaN,1,0.0,1,1,1,1.0,1.0,239745,2,residual,0.0
1,candidate_00,34.821391,NaN,1,0.0,1,1,1,1.0,1.0,35649,1,direct,0.0


,candidate_id,raw_winner,mean_paired_degradation,paired_standard_error,paired_one_se_eligible,paired_folds
0,candidate_01,candidate_01,0.000000,0.0,True,1
1,candidate_00,candidate_01,23.663212,0.0,False,1


,candidate_id,formulation,units,dense_units,dropout,learning_rate,l2_strength,num_layers,huber_delta,loss_name,...,source_fingerprint,n,mae,rmse,r2,smape_percent,best_epoch,epochs_recorded,parameters,training_seconds
0,candidate_00,direct,64,32,0.2,0.00100,0.0000,1,2.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,8469,34.821391,61.815196,0.215888,44.424033,1,1,35649,11.856687
1,candidate_01,residual,128,64,0.2,0.00056,0.0001,2,2.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,8469,11.158179,22.323596,0.897738,21.132148,1,1,239745,19.591850


{'architecture': 'lstm',
 'objective': 'mean_rolling_validation_mae',
 'values': {'candidate_id': 'candidate_01',
  'formulation': 'residual',
  'units': 128,
  'dense_units': 64,
  'dropout': 0.2,
  'learning_rate': 0.000559702206464,
  'l2_strength': 0.0001,
  'num_layers': 2,
  'huber_delta': 2.0,
  'loss_name': 'huber',
  'pinball_tau': 0.5,
  'optimizer_name': 'adam',
  'weight_decay': 0.0,
  'clipnorm': 1.0,
  'use_ema': True,
  'ema_momentum': 0.99,
  'batch_size': 256,
  'source': 'candidate13_anchor'},
 'selected_feature_policy': 'multivariate_dynamics',
 'selected_optimization_recipe': {'candidate_id': 'recipe_02',
  'formulation': 'residual',
  'units': 128,
  'dense_units': 64,
  'dropout': 0.2,
  'learning_rate': 0.000559702206464,
  'l2_strength': 0.0001,
  'num_layers': 2,
  'huber_delta': 2.0,
  'loss_name': 'huber',
  'pinball_tau': 0.5,
  'optimizer_name': 'adam',
  'weight_decay': 0.0,
  'clipnorm': 1.0,
  'use_ema': True,
  'ema_momentum': 0.99,
  'batch_size': 256,

## Final validation selection and fixed official model

**Decision.** Average seeds within each fold and compare every configuration directly
with the raw winner. Simplicity is used only when the mean paired degradation is no
greater than its paired fold-level standard error. A non-basic configuration must beat
the official basic LSTM on mean MAE and at least two folds before the benchmark can be
opened. The selected configuration is trained once with seed 42 for the median
validation-derived epoch budget. **Trade-off.** The gate can reject an apparent small
gain, but prevents a benchmark-driven retry and makes the final claim auditable.


In [5]:
final_selection = json.loads(
    (PATHS.models / f"final_model_selection{suffix}.json").read_text(encoding="utf-8")
)
validation_gate = json.loads(
    (PATHS.models / f"focused_lstm_validation_gate{suffix}.json").read_text(encoding="utf-8")
)
display(final_selection)
display(validation_gate)
print("Rolling tuning and single-model selection complete.")


{'architecture': 'lstm',
 'selected_candidate': {'candidate_id': 'candidate_01',
  'formulation': 'residual',
  'units': 128,
  'dense_units': 64,
  'dropout': 0.2,
  'learning_rate': 0.000559702206464,
  'l2_strength': 0.0001,
  'num_layers': 2,
  'huber_delta': 2.0,
  'loss_name': 'huber',
  'pinball_tau': 0.5,
  'optimizer_name': 'adam',
  'weight_decay': 0.0,
  'clipnorm': 1.0,
  'use_ema': True,
  'ema_momentum': 0.99,
  'batch_size': 256,
  'source': 'candidate13_anchor'},
 'selected_seed': 42,
 'epoch_budget': 1,
 'selection_metric': 'mean_rolling_validation_mae',
 'mean_validation_mae': 11.158179018198776,
 'validation_standard_error': 0.0,
 'selected_feature_policy': 'multivariate_dynamics',
 'selected_optimization_recipe': {'candidate_id': 'recipe_02',
  'formulation': 'residual',
  'units': 128,
  'dense_units': 64,
  'dropout': 0.2,
  'learning_rate': 0.000559702206464,
  'l2_strength': 0.0001,
  'num_layers': 2,
  'huber_delta': 2.0,
  'loss_name': 'huber',
  'pinball_tau'

{'passed': True,
 'bypassed_for_smoke': True,
 'checks': {'non_basic_configuration': True,
  'lower_mean_mae_than_basic_lstm': True,
  'wins_required_fold_averages': True,
  'complete_search': True},
 'selected_candidate': 'candidate_01',
 'selected_source': 'candidate13_anchor',
 'selected_mean_mae': 11.158179018198776,
 'basic_lstm_mean_mae': 29.575890571109127,
 'fold_wins': 1,
 'required_fold_wins': 1,
 'paired_fold_metrics': [{'fold': 'fold_1',
   'tuned_mae': 11.158179018198776,
   'basic_mae': 29.575890571109127}],
 'source_fingerprint': 'c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab364434d62d892fdac46'}

Rolling tuning and single-model selection complete.
